# 09 — Gate 2b: MGA reference run (spec v0.10 §7; estimator `mga_maxham_v1`)

The re-operationalized within-cell estimator on the reference cell
(`cell_s0_ssp585_theta5`): certified **anchor** (with the t=0 tails-equivalence assert), then
**k=50 maximally-diverse members** of the g-band at **g = 5%**, plus **f(g) probes at 2% and
10%** — each g on a fresh copy of the compiled model with one appended band-wall row
(objective ≤ (1+g)·z*). Direct Gurobi calls (`mga_core.R`, toy-verified); ~1 min/iterate ⇒
**~2.5–3 h total**; resumable per g; **live internet** (WLS) throughout. Run 08 first
(tails + scenarios_v2 + manifest). Kernel: `R (y2y)`.

In [ ]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
ctx   <- pr_setup(mpath, PROJ)

In [ ]:
# ---- ingest (banner must read 10 continuous + 40 EFG -- the tails are in) ------------------
ctx <- modifyList(ctx, pr_ingest(ctx))
stopifnot("tails missing from the stack -- run 08_gate2a_tails first" = ctx$n_cont == 10)
ctx <- modifyList(ctx, pr_planning_units(ctx))

In [ ]:
# ---- S0 from scenarios_v2 (weights unchanged; tail targets 0.0 = mathematically absent) ----
sc  <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/scenarios_v2.json"))
S0W <- sc$S0_balanced$weights
S0T <- sc$S0_balanced$targets
cat(sprintf("scenarios_v2 (%s, estimator %s): S0 targets:\n",
            sc$`_meta`$spec_version, sc$`_meta`$estimator))
for (nm in names(S0T)) cat(sprintf("  %-32s %.3f\n", nm, as.numeric(S0T[[nm]])))

ctx <- pr_override(ctx, targets = S0T, feature_weight_multipliers = S0W,
                   results_subdir = "iter10_y2y_s0_mga")   # scratch tag; outputs go to runs/
ctx <- modifyList(ctx, pr_weights(ctx))
ctx <- modifyList(ctx, pr_targets(ctx))
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params
cm <- mga_compile(ctx)

In [ ]:
# ---- anchor: certified optimum of the v2-stack S0 (t=0 tails must change NOTHING) ----------
anchor <- mga_anchor(cm, opt_gap = 1e-4)
Z_ITER9 <- 5.362813   # pool-best certified optimum on the v1 stack (results_log R6.2)
stopifnot("anchor deviates from iter9 -- t=0 tails are NOT inert; STOP" =
            abs(anchor$z - Z_ITER9) < 1e-3)
cat(sprintf("t=0 equivalence PROVEN: anchor %.6f vs iter9 %.6f (delta %.1e)\n",
            anchor$z, Z_ITER9, abs(anchor$z - Z_ITER9)))

OUT <- file.path(PROJ, "analyses/y2y/runs/cell_s0_ssp585_theta5")
dir.create(OUT, recursive = TRUE, showWarnings = FALSE)
# anchor raster + meta
r <- terra::rast(ctx$cost); v <- rep(NA_integer_, terra::ncell(r))
v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
terra::writeRaster(r, file.path(OUT, "anchor.tif"), overwrite = TRUE, datatype = "INT1U",
                   NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
jsonlite::write_json(list(
  cell_id = "cell_s0_ssp585_theta5", estimator = "mga_maxham_v1",
  anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
  anchor_runtime_s = anchor$runtime, iter9_reference = Z_ITER9,
  k = 50, g_levels = c(0.02, 0.05, 0.10), opt_gap = 1e-4,
  mip_gap_dist = 0.01, time_limit_iter = 900,
  scenarios_meta = sc$`_meta`, created_utc = format(Sys.time(), tz = "UTC")),
  file.path(OUT, "gate2b_meta.json"), auto_unbox = TRUE, pretty = TRUE)
cat("anchor.tif + gate2b_meta.json written\n")

In [ ]:
# ---- MGA sweep 1/3: g = 5%, k = 50 (the headline band; ~50 min) ----------------------------
if (file.exists(file.path(OUT, "mga_g05.tif"))) {
  cat("g05 already generated -- skipped\n")
} else {
  gen05 <- mga_generate(cm, anchor, g = 0.05, k = 50)
  mga_write(gen05, cm, ctx$cost, OUT, "g05")
}

In [ ]:
# ---- MGA sweep 2/3: g = 2% probe (~50 min) -------------------------------------------------
if (file.exists(file.path(OUT, "mga_g02.tif"))) {
  cat("g02 already generated -- skipped\n")
} else {
  gen02 <- mga_generate(cm, anchor, g = 0.02, k = 50)
  mga_write(gen02, cm, ctx$cost, OUT, "g02")
}

In [ ]:
# ---- MGA sweep 3/3: g = 10% probe (~50 min) ------------------------------------------------
if (file.exists(file.path(OUT, "mga_g10.tif"))) {
  cat("g10 already generated -- skipped\n")
} else {
  gen10 <- mga_generate(cm, anchor, g = 0.10, k = 50)
  mga_write(gen10, cm, ctx$cost, OUT, "g10")
}

In [ ]:
# ---- run summary ---------------------------------------------------------------------------
for (tag in c("g02", "g05", "g10")) {
  f <- file.path(OUT, sprintf("certificates_%s.csv", tag))
  if (!file.exists(f)) { cat(sprintf("%s: not run\n", tag)); next }
  d <- read.csv(f)
  cat(sprintf("%s: %d members | band all OK: %s | ham(anchor) %s-%s | %d dup | %d time-limited | %.1f min\n",
              tag, nrow(d), all(d$band_ok),
              format(min(d$hamming_to_anchor), big.mark = ","),
              format(max(d$hamming_to_anchor), big.mark = ","),
              sum(d$duplicate), sum(d$status == "TIME_LIMIT"), sum(d$runtime_s) / 60))
}
cat("\nnext: analyses/y2y/10_gate2b_analysis.ipynb (kernel y2y-geo)\n")